# Milan traffic forecasting — LSTM hyperparameter sweep

Runs the staged LSTM search on a GPU and writes the selected configuration back
into `results/tables/selected_hyperparameters.json`, merging with the harmonic
ARIMA and LightGBM selections already made locally.

### Why this one model runs here

The sweep was first run on the development machine and abandoned after 16 hours
with 2 of 5 stages complete:

| Stage | Candidates | Wall time |
|---|---:|---:|
| `sequence_length` | 4 | 36 min |
| `capacity` | 6 | **15 h** |
| remaining three | 12 | ~6 days, projected |

The winning capacity fit alone took 13.8 h, and the process held an average of
0.88 of 8 cores the whole time. That is the signature of a sequential
bottleneck rather than an undersized machine: a 288-step recurrence over
64x128 matrices cannot be spread across CPU threads, because each step depends
on the one before it. A GPU parallelises the batch dimension instead, which is
the dimension that is actually wide here.

Harmonic ARIMA and LightGBM are not affected and are tuned locally — together
they take under ten minutes.

### Before running

1. **Accelerator → GPU** (T4 or P100). The notebook asserts CUDA is present and
   stops rather than silently spending hours on the CPU.
2. **Settings → Internet → On**, needed for the `git clone` and `pip install`.
3. No Kaggle Dataset needs attaching. `data/processed/selected_series.parquet`
   and `results/tables/selected_areas.json` are committed, and tuning reads only
   the one study area — so a clean clone is the entire input.
4. Prefer **Save & Run All (Commit)**: `/kaggle/working` is only persisted by a
   committed version.

### What to bring back

Two files, from the committed version's output:
`results/tables/selected_hyperparameters.json` and `results/experiments.csv`.

## 1. Settings

In [1]:
# Point this at your own fork/clone before running.
REPO_URL = "https://github.com/Mahamatbt/milan_traffic_forecasting.git"
BRANCH = "main"

# Only the LSTM is tuned here. The other two are selected locally and their
# entries in selected_hyperparameters.json are preserved, not overwritten.
MODELS = ("lstm",)

## 2. Clone the repository and install dependencies

In [2]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(REPO_DIR / "requirements-kaggle.txt")],
    check=True,
)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Drop any previously imported src.* modules. Re-running this cell replaces the
# files on disk, but sys.modules still holds the code objects compiled from the
# old ones, so the kernel would keep executing the previous version -- visible
# as a traceback whose line numbers land on docstrings.
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]
importlib.invalidate_caches()

print("repo:", subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True).stdout.strip())

# Dynamic, because the package only exists once the clone above has run.
_check = importlib.import_module("src.models.lstm")
assert hasattr(_check, "resolve_device"), (
    "stale src/ still loaded: restart the kernel "
    "(Run -> Restart & Clear Cell Outputs), then run all cells again"
)
print("src version: OK")

Cloning into '/kaggle/working/repo'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 28.1 MB/s eta 0:00:00
repo: 2b682c7 Phase 5: three models, tuning for harmonic ARIMA and LightGBM
src version: OK


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-polars-cu12 26.2.1 requires polars<1.36,>=1.30, but you have polars 1.20.0 which is incompatible.


## 3. Confirm the GPU is actually present

The whole reason this notebook exists is the device. Running it on a CPU
accelerator would not fail — it would quietly take days — so the assertion is
worth more than the convenience of falling back.

In [3]:
import torch

from src.models.lstm import resolve_device

device = resolve_device("auto")
assert device.type == "cuda", (
    f"resolved device is {device!r}, not CUDA. Set Accelerator -> GPU in the "
    "sidebar and restart. On a CPU this sweep takes roughly six days."
)
print("device        :", device)
print("gpu           :", torch.cuda.get_device_name(0))
print("torch         :", torch.__version__)
print("capability    :", torch.cuda.get_device_capability(0))

device        : cuda
gpu           : Tesla T4
torch         : 2.10.0+cu128
capability    : (7, 5)


## 4. Configuration and hardware record

`src.config` detects the Kaggle session and layers `config/kaggle.yaml` over the
defaults. The hardware snapshot is recorded before any fitting, because the
training times reported in the results table are only meaningful alongside the
device that produced them.

In [4]:
from src.config import load_config
from src.timing import describe_environment, record_hardware

config = load_config()
config.paths.mkdirs()

assert config.is_kaggle, f"expected the Kaggle overrides, got env={config.env!r}"

env = record_hardware(config.paths.environment_json)
print(describe_environment(env))

x86_64 (2 physical / 4 logical cores), 31.3 GB RAM, GPU: Tesla T4 (15 GB), Tesla T4 (15 GB), Linux 6.12.90+, Python 3.12.13


## 5. Stage the committed inputs

Tuning reads three things that live in the repository rather than in
`/kaggle/working`: the extracted series, the study-area selection, and the
existing experiment log and selections to append to. They are copied into the
configured paths so `src.train` runs here exactly as it does locally, with no
Kaggle-specific branching in `src/`.

Copying the experiment log in is what keeps it append-only across machines: the
rows written here continue the local ones rather than starting a new file.

In [5]:
import shutil
from pathlib import Path

staged = [
    (REPO_DIR / "data/processed/selected_series.parquet",
     config.paths.processed / "selected_series.parquet"),
    (REPO_DIR / "results/tables/selected_areas.json",
     config.paths.tables / "selected_areas.json"),
    (REPO_DIR / "results/tables/selected_hyperparameters.json",
     config.paths.tables / "selected_hyperparameters.json"),
    (REPO_DIR / "results/experiments.csv", config.paths.experiments_csv),
]

for source, destination in staged:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.exists():
        shutil.copy2(source, destination)
        print(f"staged  {source.name:<34} -> {destination}")
    else:
        print(f"absent  {source.name:<34} (will be created)")

selections = config.paths.tables / "selected_hyperparameters.json"
assert selections.exists(), (
    "selected_hyperparameters.json is not in the clone. Run the local tuning "
    "first (`python run.py train --models harmonic_arima,lightgbm`) and commit "
    "the result, so this run has something to merge into."
)

staged  selected_series.parquet            -> /kaggle/working/processed/selected_series.parquet
staged  selected_areas.json                -> /kaggle/working/results/tables/selected_areas.json
staged  selected_hyperparameters.json      -> /kaggle/working/results/tables/selected_hyperparameters.json
staged  experiments.csv                    -> /kaggle/working/results/experiments.csv


## 6. Run the sweep

Five stages, searched one axis at a time and carrying the winner forward:
sequence length, capacity, optimisation, regularisation, and a calendar-feature
ablation. Every candidate appends its own row to the experiment log with the
numbers that justified keeping or rejecting it — not only the stage winner.

In [6]:
import json
import time

from src.train import train_all

started = time.perf_counter()
selected = train_all(config, models=MODELS)
elapsed = time.perf_counter() - started

print(f"\nsweep wall time: {elapsed / 60:.1f} min")
print(json.dumps(selected.get("lstm", {}), indent=2))

tuning on area 5161 (rank 1), selecting on validation MAE
models this run: lstm
train        5472 points  2013-11-01 .. 2013-12-08  [0:5472]
validation   1008 points  2013-12-09 .. 2013-12-15  [5472:6480]
test         1008 points  2013-12-16 .. 2013-12-22  [6480:7488]
stress       1440 points  2013-12-23 .. 2014-01-01  [7488:8928]
keeping existing selections for: harmonic_arima, lightgbm

--- lstm, area 5161: sequence_length (How much history the window carries) ---
  sequence_length=36                       MAE   132.47  MASE 0.381  (7s, best epoch 5)
  sequence_length=72                       MAE   136.04  MASE 0.392  (5s, best epoch 10)
  sequence_length=144                      MAE   122.29  MASE 0.352  (7s, best epoch 17)
  sequence_length=288                      MAE   138.26  MASE 0.398  (3s, best epoch 2)
  -> {'sequence_length': 144}, validation MAE 122.29

--- lstm, area 5161: capacity (Hidden size and depth) ---
  hidden_size=32, num_layers=1             MAE   129.87  MASE 0

## 7. Review what the search decided

The per-candidate rows are the evidence for the selection. A stage whose spread
across the axis is small is a stage whose parameter did not matter — which is
worth reporting, not hiding.

In [7]:
import polars as pl

rows = pl.read_csv(config.paths.experiments_csv)
lstm_rows = rows.filter(pl.col("model") == "lstm")

print(f"{len(lstm_rows)} LSTM rows logged\n")
with pl.Config(fmt_str_lengths=60, tbl_rows=60):
    print(
        lstm_rows.select("stage", "valid_mae", "valid_mase", "n_params", "train_wall_s")
        .sort("valid_mae")
    )

29 LSTM rows logged

shape: (29, 5)
┌─────────────────────────────┬────────────┬────────────┬──────────┬──────────────┐
│ stage                       ┆ valid_mae  ┆ valid_mase ┆ n_params ┆ train_wall_s │
│ ---                         ┆ ---        ┆ ---        ┆ ---      ┆ ---          │
│ str                         ┆ f64        ┆ f64        ┆ i64      ┆ f64          │
╞═════════════════════════════╪════════════╪════════════╪══════════╪══════════════╡
│ regularisation_candidate    ┆ 98.215048  ┆ 0.28272    ┆ 202369   ┆ 12.119       │
│ regularisation              ┆ 98.215048  ┆ 0.28272    ┆ 202369   ┆ 12.119       │
│ calendar_ablation_candidate ┆ 98.215048  ┆ 0.28272    ┆ 202369   ┆ 12.149       │
│ calendar_ablation           ┆ 98.215048  ┆ 0.28272    ┆ 202369   ┆ 12.149       │
│ capacity                    ┆ 103.074595 ┆ 0.296709   ┆ 202369   ┆ 49678.931    │
│ optimisation_candidate      ┆ 105.35957  ┆ 0.303286   ┆ 202369   ┆ 8.788        │
│ optimisation                ┆ 105.3595

## 8. Confirm what to download

These two files are the output of this notebook. Everything else here is
reproducible from them plus the repository.

In [8]:
for path in (config.paths.tables / "selected_hyperparameters.json",
             config.paths.experiments_csv):
    size = path.stat().st_size if path.exists() else 0
    print(f"{'OK ' if size else 'MISSING'} {path}  ({size:,} bytes)")

print("\nDownload both from the committed version's Output tab, replace the")
print("local copies, and commit them. The final runs read selected_hyperparameters.json.")

OK  /kaggle/working/results/tables/selected_hyperparameters.json  (949 bytes)
OK  /kaggle/working/results/experiments.csv  (83,898 bytes)

Download both from the committed version's Output tab, replace the
local copies, and commit them. The final runs read selected_hyperparameters.json.
